In [1]:
from datetime import date
import pandas as pd
import numpy as np
import utils.fetch_dbs
import utils.bcutils
import json
import mysql.connector
from pathlib import Path

ModuleNotFoundError: No module named 'utils'

In [ ]:
import importlib
importlib.reload(utils.fetch_dbs)
importlib.reload(utils.bcutils)

In [ ]:
datasets = pd.read_csv('hdca_reprocessing - accessions.csv')
libraries = pd.read_csv('libraries_lists/hdca_v2_libraries260602.csv')

pid='hu_2025_amnion'
author_path='/lustre/scratch124/cellgen/haniffa/users/ar32/fresh_public_object_curations/output_files/hu_2025_amnion/hu_2025_amnion_scRNA_main.h5ad'

# hu_2025_amnion

In [ ]:
aobs = utils.bcutils.get_df(author_path,'/obs')
aobs

In [ ]:
libs = libraries[libraries.publication_id==pid]
libs.index = libs.library_id
libs

In [ ]:
# laod reprocessed barcodes from irods
rbc = utils.fetch_dbs.read_gzipped_irods_files_as_set(libs.irods_path+'/output/GeneFull/filtered/barcodes.tsv.gz')
rbc = dict(zip(libs.library_id,rbc))

In [ ]:
# get author barcodes
abc = aobs.groupby('obs_unit_id')['barcode'].apply(set).to_dict()
overlap_matrix, jaccard_matrix, keys_a, keys_r, matches  = utils.bcutils.compute_overlap_matrices(abc,rbc)
matches['author_sample_id'] = libs.loc[matches.key_b,'author_sample_id'].tolist()
matches['dataset_acc'] = libs.loc[matches.key_b,'dataset_acc'].tolist()
matches

In [ ]:
# check that author_ids match between author object and reprocessed data
(matches.key_a == matches.author_sample_id).value_counts()

In [ ]:
# are there any bad overlaps
matches[matches['szymkiewicz–simpson']<0.5]

# Assign library ids to author obs
proceed only if everything above is ok

In [ ]:
matches.index = matches.key_a

In [ ]:
aobs['publication_id'] = pid
aobs['dataset_acc'] = matches.loc[aobs.obs_unit_id,'dataset_acc'].tolist()
aobs['library_id'] = matches.loc[aobs.obs_unit_id,'key_b'].tolist()
aobs['author_sample_id'] = matches.loc[aobs.obs_unit_id,'author_sample_id'].tolist()
aobs

In [ ]:
aobs.to_csv('../author_obs/'+Path(author_path).name.split(".")[0]+".csv")